# 03 – Baseline ALS và đánh giá

Notebook chạy trọn quy trình: **tải dữ liệu → lọc → chia train/test → baseline → ALS → bảng kết quả → chỉnh siêu tham số**, theo `docs/evaluation_protocol.md`.

- Nếu chưa có `data/processed/sample_interactions.parquet` (Thy chưa giao), notebook dùng **dữ liệu giả** để chạy thử. Kết quả trên dữ liệu giả **chỉ để kiểm tra code**, không dùng để kết luận.
- Cần `pip install -r requirements.txt` (có `implicit`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # để import src.*
import numpy as np, pandas as pd
from src.split import split_interactions
from src.evaluate import evaluate
from src.baselines import popularity_recs, random_recs
from src.model import build_matrix, train_als, recommend_all

SEED, MIN_USER, MIN_ITEM_USERS, TEST_RATIO, KS = 42, 5, 2, 0.2, (10, 20)
DATA = "../data/processed/sample_interactions.parquet"

## 1. Tải dữ liệu

In [ ]:
if os.path.exists(DATA):
    inter = pd.read_parquet(DATA)
    SYNTHETIC = False
else:
    from src.synthetic import make_synthetic
    inter = make_synthetic(seed=SEED)
    SYNTHETIC = True
print("DỮ LIỆU GIẢ" if SYNTHETIC else "Dữ liệu thật", "| dòng:", len(inter))
assert {"user_id", "track_id", "playcount"} <= set(inter.columns)
inter = inter.groupby(["user_id", "track_id"], as_index=False)["playcount"].sum()
inter = inter[inter["playcount"] > 0]
inter.head()

## 2. Lọc và thống kê
Ghi lại các con số này vào báo cáo.

In [ ]:
n_users_item = inter.groupby("track_id")["user_id"].transform("nunique")
inter = inter[n_users_item >= MIN_ITEM_USERS]
n_inter_user = inter.groupby("user_id")["track_id"].transform("size")
inter = inter[n_inter_user >= MIN_USER]

n_u, n_i, n_r = inter.user_id.nunique(), inter.track_id.nunique(), len(inter)
print(f"user={n_u:,}  bài={n_i:,}  tương tác={n_r:,}  độ thưa={1 - n_r/(n_u*n_i):.4%}")
print(inter.playcount.describe().round(2))

## 3. Chia train / test
Mặc định `random_holdout` (không cần timestamp). Nếu dữ liệu có `timestamp`, đổi sang `temporal`.

In [ ]:
STRATEGY = "temporal" if ("timestamp" in inter.columns and inter.timestamp.notna().any()) else "random_holdout"
train, test = split_interactions(inter, STRATEGY, TEST_RATIO, MIN_USER, SEED)
print(STRATEGY, "| train:", len(train), "| test:", len(test), "| user test:", test.user_id.nunique())

## 4. Baseline: Popularity và Random

In [ ]:
test_users = test.user_id.unique().tolist()
N = max(KS)
results = {}
results["Random"]     = evaluate(random_recs(train, test_users, N, SEED), test, KS, n_items=n_i)
results["Popularity"] = evaluate(popularity_recs(train, test_users, N), test, KS, n_items=n_i)
pd.DataFrame(results).T.round(4)

## 5. ALS
`build_matrix` đã nhân `alpha` vào ma trận (confidence), nên `train_als` dùng `alpha=1.0`.
**Kiểm tra chiều ma trận:** `fit()` của `implicit` ≥ 0.5 nhận user × item; bản cũ nhận item × user. Nếu kết quả tệ hơn Popularity một cách vô lý, kiểm tra điều này đầu tiên.

In [ ]:
mat, u_map, t_map = build_matrix(train, transform="log1p", alpha=40.0)
model = train_als(mat, factors=64, regularization=0.01, iterations=20, seed=SEED)
recs = recommend_all(model, mat, u_map, t_map, test_users, n=N)
results["ALS (f=64, a=40)"] = evaluate(recs, test, KS, n_items=n_i)
pd.DataFrame(results).T.round(4)

## 6. Chỉnh siêu tham số trên validation
Tách validation từ **train** (cùng cách chia), chọn tham số theo NDCG@10, **không** nhìn test.

In [ ]:
tr2, val = split_interactions(train, STRATEGY, 0.125, MIN_USER, SEED)
val_users = val.user_id.unique().tolist()
rows = []
for factors in (32, 64, 128):
    for reg in (0.01, 0.1):
        for alpha in (10, 40):
            m2, um2, tm2 = build_matrix(tr2, "log1p", alpha)
            mdl = train_als(m2, factors=factors, regularization=reg, iterations=15, seed=SEED)
            r = evaluate(recommend_all(mdl, m2, um2, tm2, val_users, n=10), val, (10,))
            rows.append(dict(factors=factors, reg=reg, alpha=alpha, **{k: r[k] for k in ("recall@10", "ndcg@10")}))
grid = pd.DataFrame(rows).sort_values("ndcg@10", ascending=False)
grid.head(8).round(4)

## 7. Huấn luyện lại với tham số tốt nhất và đánh giá test (một lần)

In [ ]:
best = grid.iloc[0]
mat, u_map, t_map = build_matrix(train, "log1p", float(best.alpha))
model = train_als(mat, factors=int(best.factors), regularization=float(best.reg), iterations=20, seed=SEED)
recs = recommend_all(model, mat, u_map, t_map, test_users, n=N)
results[f"ALS tuned (f={int(best.factors)}, reg={best.reg}, a={best.alpha})"] = evaluate(recs, test, KS, n_items=n_i)
final = pd.DataFrame(results).T.round(4)
final

## 8. Kết luận (điền sau khi chạy trên dữ liệu thật)
- ALS so với Popularity: ...
- Tham số tốt nhất: ...
- Hạn chế (cold-start, thiên lệch phổ biến, chưa dùng mood): ...
- Bước tiếp theo: kết hợp `track_mood_labels.parquet` để lọc/xếp lại theo mood.